### Reference: Apache Airflow Setup in WSL with `uv`
**Local Development Environment Config for Airflow**

This notebook serves as a technical reference for the local installation and configuration of Apache Airflow within a WSL (Windows Subsystem for Linux) environment. The goal was to integrate Airflow in the repo as an MLOps tool without compromising the existing dependency requirements of the core ML sub-packages (Tensorflow, Computer Vision).

We had dependency conflict with the initial `uv add apache-airflow` command since
* **Core Project Requirements:** NumPy `>= 2.4.1` (Required for modern Tensorflow/CV performance).
* **Airflow 2.10.4 Requirements:** NumPy `== 1.26.4` (Airflow is not yet compatible with NumPy 2.x). 

Therefore, we created an isolated Airflow env here.

The following steps were performed in the WSL terminal to establish the environment.
1. Environment Creation
We created a dedicated virtual environment specifically for Airflow to prevent conflict with the main `uv.lock`.

```bash
# From ~/workspace/ml_toolbox/mlops/Airflow
uv venv .venv_airflow --python 3.12
source .venv_airflow/bin/activate
```

2. Airflow must be installed with a constraint file to ensure stability.

```bash
uv pip install "apache-airflow==2.10.4" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.10.4/constraints-3.12.txt"
```
3. Environment Initialization
We set the `AIRFLOW_HOME` to the current directory to keep the repository portable and initialized the metadata database.

```bash
export AIRFLOW_HOME=$(pwd)

# Initialize the SQLite DB
airflow db init

# Create Admin User (Replace with your chosen credentials)
airflow users create \
    --username ***** \
    --firstname ***** \
    --lastname ***** \
    --role Admin \
    --email admin@example.com \
    --password *****
```

4. Current Repository Architecture
The final structure ensures that the **Orchestrator** (Airflow) and the **Workloads** (Tensorflow/CV) are decoupled.

```text
ml_toolbox/
├── .venv/                  <-- Main ML Environment (NumPy 2.x)
├── mlops/
│   ├── Airflow/            <-- AIRFLOW_HOME
│   │   ├── .venv_airflow/  <-- Airflow Environment (NumPy 1.x)
│   │   ├── airflow.db
│   │   └── dags/           <-- Target for Workflow scripts
│   ├── 00_airflow_setup_reference.ipynb
│   └── 01_airflow_fundamentals.ipynb
├── Tensorflow/
└── ...../
```

5. Maintenance & Operations
To start the environment in a new session, run:

1. `cd ~/workspace/ml_toolbox/MLOps/Airflow`
2. `source .venv_airflow/bin/activate`
3. `export AIRFLOW_HOME=$(pwd)`
4. `airflow webserver --port 8080` (Tab 1)
5. `airflow scheduler` (Tab 2)

**Troubleshooting Tip:** If `localhost:8080` is unreachable, check if WSL port forwarding is active or if another process is using the port.